# Thai Next-Word Prediction — KaggleHub + HSE Thai Corpus

`kagglehub` → HSE Thai Corpus (CSV) → Thai tokenization → streaming `tf.data` → causal Transformer Encoder → next-word prediction → **F1 evaluation** → 5-word generation → TFLite.

This notebook does **not** use `files.upload()`; the dataset is downloaded directly with `kagglehub`.

In [ ]:
# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================

import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pythainlp",
    "kagglehub",
    "scikit-learn"
])

print("Dependencies installed successfully.")

In [ ]:
# ============================================================
# 2. IMPORTS + CONFIG
# ============================================================
# NOTE: TensorFlow is imported later (section 11), AFTER the parallel
# tokenization, because forking worker processes after TF is loaded can hang.

import os
import re
import json
import random
from collections import Counter

import numpy as np
import pandas as pd

import kagglehub

from pythainlp.tokenize import word_tokenize

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# ------------------------------------------------------------
# Model configuration
# ------------------------------------------------------------

VOCAB_SIZE = 20_000
MAX_LEN = 30

EMBED_DIM = 128
NUM_HEADS = 4
FF_DIM = 256
DROPOUT = 0.25

BATCH_SIZE = 512          # was 128 -> 4x fewer steps, much better GPU use
EPOCHS = 30
LEARNING_RATE = 2e-3      # scaled up together with the batch size
WEIGHT_DECAY = 1e-4

# ------------------------------------------------------------
# Data configuration
# ------------------------------------------------------------

# Rows read from EACH csv file (None = read everything).
MAX_ROWS_PER_FILE = 200_000

# Rows used to count word frequencies for the vocabulary
VOCAB_SCAN_ROWS = 100_000

# Training examples. Use 200_000 for a quick first run.
MAX_TRAIN_SAMPLES = 1_000_000
MAX_VAL_SAMPLES = 100_000

VAL_FRACTION = 0.05

# CPU processes used for Thai tokenization (newmm is CPU-bound pure Python)
NUM_WORKERS = max(1, os.cpu_count() or 1)

# Works on Kaggle, Colab and local machines
if os.path.isdir("/kaggle/working"):
    OUTPUT_DIR = "/kaggle/working"
elif os.path.isdir("/content"):
    OUTPUT_DIR = "/content"
else:
    OUTPUT_DIR = os.getcwd()

print("CPU workers:", NUM_WORKERS)
print("Output dir:", OUTPUT_DIR)

In [ ]:
# ============================================================
# 3. DOWNLOAD DATASET
# ============================================================

dataset_path = kagglehub.dataset_download("rtatman/hse-thai-corpus")

print("Dataset path:", dataset_path)

csv_files = []

for root, dirs, files in os.walk(dataset_path):
    for file in sorted(files):
        full_path = os.path.join(root, file)
        print(full_path)

        if file.lower().endswith(".csv"):
            csv_files.append(full_path)

if not csv_files:
    raise FileNotFoundError("No CSV file found in the dataset folder.")

In [ ]:
# ============================================================
# 4. LOAD CSV + INSPECT SCHEMA
# ============================================================
# (replaces the old `hf_dataset`, which was never defined -> NameError)

frames = []

for path in csv_files:
    part = pd.read_csv(
        path,
        nrows=MAX_ROWS_PER_FILE,
        on_bad_lines="skip"
    )
    part["source_file"] = os.path.basename(path)
    frames.append(part)

    print(f"{os.path.basename(path)}: {len(part):,} rows")

df = pd.concat(frames, ignore_index=True)

# Shuffle so train / validation contain both corpora
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print("\nColumns:")
print(df.dtypes)

print("\nNumber of rows:", f"{len(df):,}")

print("\nFirst example:")
print(df.iloc[0].to_dict())

## 5. Automatically find the text column

The notebook loads the CSV files with pandas and picks the string column with the longest average text.

If the inspection cell shows a specific column you prefer, set `TEXT_COLUMN` manually.

In [ ]:
# ============================================================
# 5. FIND TEXT COLUMN
# ============================================================

TEXT_COLUMN = None   # set manually if the auto-detection picks the wrong one

def detect_text_column(frame, manual_column=None):

    if manual_column is not None:
        if manual_column not in frame.columns:
            raise ValueError(
                f"Column '{manual_column}' not found. "
                f"Available columns: {list(frame.columns)}"
            )
        return manual_column

    best_name = None
    best_len = -1.0

    for name in frame.columns:

        if name == "source_file":
            continue

        # works for both pandas 2.x (object) and pandas 3.x (str)
        if pd.api.types.is_numeric_dtype(frame[name]):
            continue

        mean_len = (
            frame[name]
            .dropna()
            .astype(str)
            .head(1000)
            .str.len()
            .mean()
        )

        if mean_len > best_len:
            best_name = name
            best_len = mean_len

    if best_name is None:
        raise ValueError(
            "Could not automatically find a text column. "
            f"Available columns: {list(frame.columns)}"
        )

    return best_name


TEXT_COLUMN = detect_text_column(df, TEXT_COLUMN)

print("Using text column:", TEXT_COLUMN)

# Remove duplicate texts BEFORE the train/validation split.
# Otherwise the same sentence can be in both sets and the model just memorizes it.
before = len(df)
df = df.dropna(subset=[TEXT_COLUMN])
df = df.drop_duplicates(subset=[TEXT_COLUMN]).reset_index(drop=True)
print(f"Removed {before - len(df):,} duplicate/empty rows ({(before - len(df)) / max(before, 1):.1%})")

texts = [
    t for t in df[TEXT_COLUMN].dropna().astype(str).tolist()
    if t.strip()
]

row_count = len(texts)

VAL_START_ROW = int(row_count * (1.0 - VAL_FRACTION))

print("Usable rows:", f"{row_count:,}")

In [ ]:
# ============================================================
# 6. TEST TEXT + THAI TOKENIZATION
# ============================================================

def clean_text(text):
    text = str(text)
    text = text.replace("\\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def thai_tokenize(text):
    text = clean_text(text)

    if not text:
        return []

    tokens = word_tokenize(
        text,
        engine="newmm",
        keep_whitespace=False
    )

    return [
        token.strip()
        for token in tokens
        if token.strip()
    ]


example = texts[0]

print("Example:")
print(example)

print("\nTokens:")
print(thai_tokenize(example)[:50])

## 7. Build vocabulary

The HSE corpus is scanned through the Hugging Face Dataset and token frequencies are counted.

Vocabulary:

- `<PAD>` = 0
- `<UNK>` = 1
- `<START>` = 2
- remaining IDs = most frequent Thai tokens

The final vocabulary is limited to 20,000 tokens.

In [ ]:
# ============================================================
# 7. BUILD VOCABULARY  (training rows only, parallel)
# ============================================================

import multiprocessing as mp

def chunked(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def parallel_map(func, chunks):
    """Runs func over chunks on all CPU cores (serial if NUM_WORKERS == 1)."""
    if NUM_WORKERS <= 1:
        for chunk in chunks:
            yield func(chunk)
        return

    ctx = mp.get_context("fork")
    with ctx.Pool(NUM_WORKERS) as pool:
        for result in pool.imap(func, chunks):
            yield result


def _count_chunk(chunk):
    c = Counter()
    for t in chunk:
        c.update(thai_tokenize(t))
    return c


counter = Counter()

scan_texts = texts[:min(VOCAB_SCAN_ROWS, VAL_START_ROW)]
CHUNK = 2_000
done = 0

for part in parallel_map(_count_chunk, chunked(scan_texts, CHUNK)):
    counter.update(part)
    done += CHUNK
    print(
        f"Processed: {min(done, len(scan_texts)):,} / {len(scan_texts):,} "
        f"| unique tokens: {len(counter):,}"
    )

print("\nVocabulary scan complete.")
print("Unique tokens:", f"{len(counter):,}")

word_to_id = {
    "<PAD>": 0,
    "<UNK>": 1,
    "<START>": 2
}

for idx, (word, _) in enumerate(
    counter.most_common(VOCAB_SIZE - 3),
    start=3
):
    word_to_id[word] = idx

id_to_word = {
    idx: word
    for word, idx in word_to_id.items()
}

print("Final vocabulary:", len(word_to_id))

In [ ]:
# ============================================================
# 8. ENCODE / EXAMPLE
# ============================================================

def encode_tokens(tokens):
    return [
        word_to_id.get(token, 1)
        for token in tokens
    ]


example_tokens = thai_tokenize(example)
example_ids = encode_tokens(example_tokens)

print("Tokens:")
print(example_tokens[:30])

print("\nIDs:")
print(example_ids[:30])

## 9. Create next-word examples (parallel, once)

For each sentence:

```text
วันนี้ | ฉัน | ไป | มหาวิทยาลัย
```

the examples are

```text
วันนี้                         → ฉัน
วันนี้ ฉัน                    → ไป
วันนี้ ฉัน ไป                 → มหาวิทยาลัย
```

with left padding up to `MAX_LEN`.

Tokenization is done **once, on all CPU cores, before training** and stored as NumPy arrays (1M × 30 int32 ≈ 120 MB), then saved to disk so a re-run skips it. This keeps the GPU busy instead of waiting for a Python generator.

In [ ]:
# ============================================================
# 9. EXAMPLE BUILDER (vectorized, parallel)
# ============================================================

from numpy.lib.stride_tricks import sliding_window_view


def _encode_chunk(chunk):
    """texts -> (contexts [N, MAX_LEN], targets [N]) with left padding."""

    xs, ys = [], []

    for t in chunk:

        ids = encode_tokens(thai_tokenize(t))
        n = len(ids)

        if n < 2:
            continue

        padded = np.concatenate([
            np.zeros(MAX_LEN, dtype=np.int32),
            np.asarray(ids, dtype=np.int32)
        ])

        # row i-1 = ids[i-MAX_LEN:i] (left-padded) -> target ids[i]
        windows = sliding_window_view(padded, MAX_LEN)[1:n]

        xs.append(windows.copy())
        ys.append(np.asarray(ids[1:], dtype=np.int32))

    if not xs:
        return (
            np.empty((0, MAX_LEN), dtype=np.int32),
            np.empty((0,), dtype=np.int32)
        )

    return np.concatenate(xs), np.concatenate(ys)


def build_arrays(text_list, max_samples):

    xs, ys = [], []
    total = 0

    for x, y in parallel_map(_encode_chunk, chunked(text_list, 1_000)):

        xs.append(x)
        ys.append(y)
        total += len(y)

        print(f"\r  examples: {min(total, max_samples):,} / {max_samples:,}", end="")

        if total >= max_samples:
            break

    print()

    X = np.concatenate(xs)[:max_samples]
    Y = np.concatenate(ys)[:max_samples]

    return X, Y

In [ ]:
# ============================================================
# 10. TRAIN / VALIDATION SPLIT
# ============================================================

print("Total rows:", f"{row_count:,}")
print("Train rows:", f"{VAL_START_ROW:,}")
print("Validation rows:", f"{row_count - VAL_START_ROW:,}")

In [ ]:
# ============================================================
# 11. BUILD ARRAYS (cached on disk) + TF.DATA
# ============================================================

import time
import hashlib

vocab_hash = hashlib.md5("|".join(word_to_id).encode("utf-8")).hexdigest()

cache_key = hashlib.md5(json.dumps([
    vocab_hash, MAX_LEN, row_count, VAL_START_ROW,
    MAX_TRAIN_SAMPLES, MAX_VAL_SAMPLES
]).encode()).hexdigest()[:12]

CACHE_FILE = os.path.join(OUTPUT_DIR, f"tokenized_{cache_key}.npz")

if os.path.exists(CACHE_FILE):

    print("Loading tokenized arrays from", CACHE_FILE)

    data = np.load(CACHE_FILE)
    X_train, y_train = data["X_train"], data["y_train"]
    X_val, y_val = data["X_val"], data["y_val"]

else:

    t0 = time.time()

    print("Building training examples...")
    X_train, y_train = build_arrays(texts[:VAL_START_ROW], MAX_TRAIN_SAMPLES)

    print("Building validation examples...")
    X_val, y_val = build_arrays(texts[VAL_START_ROW:], MAX_VAL_SAMPLES)

    np.savez(
        CACHE_FILE,
        X_train=X_train, y_train=y_train,
        X_val=X_val, y_val=y_val
    )

    print(f"Done in {time.time() - t0:.0f}s, cached to {CACHE_FILE}")

print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)

# ------------------------------------------------------------
# TensorFlow is imported only now (after the worker processes are done)
# ------------------------------------------------------------

import tensorflow as tf

tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

train_ds = (
    tf.data.Dataset
    .from_tensor_slices((X_train, y_train))
    .shuffle(len(y_train), seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset
    .from_tensor_slices((X_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("Steps per epoch:", len(train_ds))

# Check that data is actually produced
for x_batch, y_batch in train_ds.take(1):

    print("X shape:", x_batch.shape)
    print("y shape:", y_batch.shape)

    print("First X:")
    print(x_batch[0].numpy())

    print("First target:")
    print(y_batch[0].numpy())

    break

# 12. Transformer Encoder with Causal Attention

The model uses causal self-attention so the representation at a position cannot use future tokens.

Architecture:

**Token IDs → Embedding + Position → Causal Transformer Encoder → Last position → Dense(20,000) → Softmax**

In [ ]:
# ============================================================
# 12. EMBEDDING + TRANSFORMER BLOCK
# ============================================================

@tf.keras.utils.register_keras_serializable(package="thai")
class TokenAndPositionEmbedding(tf.keras.layers.Layer):

    def __init__(self, vocab_size, max_len, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.max_len = max_len
        self.embed_dim = embed_dim

        self.token_emb = tf.keras.layers.Embedding(
            vocab_size, embed_dim, name="token_embedding"
        )
        self.pos_emb = tf.keras.layers.Embedding(
            max_len, embed_dim, name="position_embedding"
        )

    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1])
        return self.token_emb(x) + self.pos_emb(positions)

    def get_config(self):
        config = super().get_config()
        config.update({
            "vocab_size": self.vocab_size,
            "max_len": self.max_len,
            "embed_dim": self.embed_dim,
        })
        return config


@tf.keras.utils.register_keras_serializable(package="thai")
class LastTimestep(tf.keras.layers.Layer):
    """Takes the representation of the last position: (B, T, D) -> (B, D)."""

    def call(self, x):
        return x[:, -1, :]


@tf.keras.utils.register_keras_serializable(package="thai")
class TransformerBlock(tf.keras.layers.Layer):

    def __init__(
        self,
        embed_dim,
        num_heads,
        ff_dim,
        dropout=0.1,
        **kwargs
    ):
        super().__init__(**kwargs)

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.dropout_rate = dropout

        self.att = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=dropout
        )

        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(ff_dim, activation="gelu"),
            tf.keras.layers.Dense(embed_dim)
        ])

        self.norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = tf.keras.layers.Dropout(dropout)
        self.dropout2 = tf.keras.layers.Dropout(dropout)

    def call(self, x, training=False):

        attention_output = self.att(
            x,
            x,
            use_causal_mask=True,
            training=training
        )

        attention_output = self.dropout1(
            attention_output,
            training=training
        )

        x = self.norm1(x + attention_output)

        ffn_output = self.ffn(x)

        ffn_output = self.dropout2(
            ffn_output,
            training=training
        )

        return self.norm2(x + ffn_output)

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "ff_dim": self.ff_dim,
            "dropout": self.dropout_rate,
        })
        return config

In [ ]:
# ============================================================
# 13. BUILD MODEL
# ============================================================

token_input = tf.keras.Input(
    shape=(MAX_LEN,),
    dtype=tf.int32,
    name="tokens"
)

x = TokenAndPositionEmbedding(
    vocab_size=len(word_to_id),
    max_len=MAX_LEN,
    embed_dim=EMBED_DIM,
    name="embedding"
)(token_input)

x = TransformerBlock(
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    ff_dim=FF_DIM,
    dropout=DROPOUT
)(x)

x = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)

# Last position predicts the next token
x = LastTimestep(name="last_timestep")(x)

x = tf.keras.layers.Dropout(DROPOUT)(x)

output = tf.keras.layers.Dense(
    len(word_to_id),
    activation="softmax",
    name="next_word"
)(x)

model = tf.keras.Model(token_input, output)

model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    ),
    loss="sparse_categorical_crossentropy",
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=5, name="top5_accuracy"
        )
    ]
)

model.summary()

In [ ]:
# ============================================================
# 13.5 SPEED CHECK  (run once before training, ~20 seconds)
# ============================================================
# Measures the input pipeline and the model separately, without
# changing any weights, so you can see WHAT is slow.

import time

print("GPU visible to TensorFlow:", tf.config.list_physical_devices("GPU"))

x_b, y_b = next(iter(train_ds))


@tf.function
def forward_backward(x, y):
    with tf.GradientTape() as tape:
        probs = model(x, training=True)
        loss = tf.reduce_mean(
            tf.keras.losses.sparse_categorical_crossentropy(y, probs)
        )
    grads = tape.gradient(loss, model.trainable_variables)
    return tf.linalg.global_norm(grads)   # scalar -> forces GPU sync


# --- model compute (forward + backward, one fixed batch) ---
for _ in range(3):
    forward_backward(x_b, y_b).numpy()    # warm-up / graph tracing

N = 30
t0 = time.time()
for _ in range(N):
    forward_backward(x_b, y_b).numpy()
compute_ms = (time.time() - t0) / N * 1000

# --- input pipeline alone ---
it = iter(train_ds)
next(it)                                   # warm-up (fills shuffle buffer)
t0 = time.time()
for _ in range(N):
    next(it)
data_ms = (time.time() - t0) / N * 1000

print(f"\nBatch size           : {BATCH_SIZE}")
print(f"Model  fwd+bwd       : {compute_ms:6.1f} ms/step")
print(f"Input pipeline       : {data_ms:6.1f} ms/step")
print(f"Steps per epoch      : {len(train_ds):,}")
print(f"Estimated epoch time : {len(train_ds) * max(compute_ms, data_ms) / 60000:.1f} min")

if not tf.config.list_physical_devices("GPU"):
    print("\n-> NO GPU: training runs on CPU. Colab: Runtime > Change runtime type > T4 GPU.")
elif data_ms > 0.7 * compute_ms:
    print("\n-> INPUT-BOUND: the data pipeline is the bottleneck.")
else:
    print("\n-> COMPUTE-BOUND: the model itself is the bottleneck (see notes below).")

In [ ]:
# ============================================================
# 14. TRAIN
# ============================================================

callbacks = [

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        min_delta=0.001,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-6,
        verbose=1
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

In [ ]:
# ============================================================
# 15. TRAINING CURVES
# ============================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(
    history.history["loss"],
    label="train"
)
plt.plot(
    history.history["val_loss"],
    label="validation"
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training / Validation Loss")
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(
    history.history["accuracy"],
    label="train"
)
plt.plot(
    history.history["val_accuracy"],
    label="validation"
)
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training / Validation Accuracy")
plt.legend()
plt.show()

## 15.5 F1 Score (validation set)

Next-word prediction is a multi-class problem with ~20,000 classes, so three F1 averages are reported:

- **Micro F1** – counts every prediction equally (for single-label multi-class this equals accuracy)
- **Macro F1** – averages F1 over every word class equally (rare words count as much as common ones → naturally low)
- **Weighted F1** – averages F1 over classes, weighted by how often each word appears

In [ ]:
# ============================================================
# 15.5 F1 SCORE
# ============================================================

from sklearn.metrics import f1_score, precision_score, recall_score


def collect_predictions(dataset, max_batches=None):
    """Runs the model batch by batch (never stores the 20k-wide probability matrix)."""

    y_true_all = []
    y_pred_all = []
    top5_hits = 0
    total = 0

    for batch_index, (x_batch, y_batch) in enumerate(dataset):

        if max_batches is not None and batch_index >= max_batches:
            break

        probs = model(x_batch, training=False)

        pred = tf.argmax(probs, axis=-1, output_type=tf.int32)
        top5 = tf.math.top_k(probs, k=5).indices

        y_np = y_batch.numpy()

        top5_hits += int(
            np.any(top5.numpy() == y_np[:, None], axis=1).sum()
        )
        total += len(y_np)

        y_true_all.append(y_np)
        y_pred_all.append(pred.numpy())

    return (
        np.concatenate(y_true_all),
        np.concatenate(y_pred_all),
        top5_hits / max(total, 1)
    )


y_true, y_pred, top5_acc = collect_predictions(val_ds)

print("Validation samples:", f"{len(y_true):,}")

results = {
    "accuracy (top-1)": float((y_true == y_pred).mean()),
    "top-5 accuracy":   top5_acc,

    "micro F1":    f1_score(y_true, y_pred, average="micro",    zero_division=0),
    "macro F1":    f1_score(y_true, y_pred, average="macro",    zero_division=0),
    "weighted F1": f1_score(y_true, y_pred, average="weighted", zero_division=0),

    "weighted precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
    "weighted recall":    recall_score(y_true, y_pred, average="weighted",    zero_division=0),
}

print()
for name, value in results.items():
    print(f"{name:20s}: {value:.4f}")

# ------------------------------------------------------------
# Sanity check: how much of the score comes from <UNK>?
# ------------------------------------------------------------

unk_ratio = float((y_true == 1).mean())
print(f"\nShare of <UNK> targets in validation: {unk_ratio:.2%}")

mask = (y_true != 0) & (y_true != 1)   # drop <PAD> and <UNK> targets

print(f"Validation samples without <UNK>/<PAD>: {mask.sum():,}")
print(f"accuracy (no UNK):    {float((y_true[mask] == y_pred[mask]).mean()):.4f}")
print(f"micro F1 (no UNK):    {f1_score(y_true[mask], y_pred[mask], average='micro',    zero_division=0):.4f}")
print(f"macro F1 (no UNK):    {f1_score(y_true[mask], y_pred[mask], average='macro',    zero_division=0):.4f}")
print(f"weighted F1 (no UNK): {f1_score(y_true[mask], y_pred[mask], average='weighted', zero_division=0):.4f}")

# ------------------------------------------------------------
# F1 of the most frequent target words
# ------------------------------------------------------------

top_classes = [
    int(c) for c, _ in Counter(y_true.tolist()).most_common(15)
]

per_class_f1 = f1_score(
    y_true,
    y_pred,
    labels=top_classes,
    average=None,
    zero_division=0
)

print("\nF1 of the 15 most frequent target words:")
print(f"{'word':<14}{'count':>8}{'F1':>8}")

true_counts = Counter(y_true.tolist())

for class_id, f1_value in zip(top_classes, per_class_f1):
    print(
        f"{id_to_word.get(class_id, '?'):<14}"
        f"{true_counts[class_id]:>8}"
        f"{f1_value:>8.3f}"
    )

In [ ]:
# ============================================================
# 16. TOP-K NEXT WORD
# ============================================================

def encode_text(text):

    tokens = thai_tokenize(text)

    return [
        word_to_id.get(
            token,
            1
        )
        for token in tokens
    ]


def predict_next_words(
    text,
    top_k=5
):

    ids = encode_text(text)

    if not ids:
        return []

    context = ids[-MAX_LEN:]

    if len(context) < MAX_LEN:
        context = (
            [0] * (MAX_LEN - len(context))
            + context
        )

    x = np.asarray(
        [context],
        dtype=np.int32
    )

    probabilities = model.predict(
        x,
        verbose=0
    )[0]

    top_ids = np.argsort(
        probabilities
    )[-top_k:][::-1]

    return [
        (
            id_to_word.get(
                int(token_id),
                "<UNK>"
            ),
            float(
                probabilities[token_id]
            )
        )
        for token_id in top_ids
    ]


def print_suggestions(
    text,
    top_k=5
):

    print("\nInput:")
    print(text)

    print("\nPredictions:")

    for i, (word, probability) in enumerate(
        predict_next_words(
            text,
            top_k
        ),
        start=1
    ):

        print(
            f"{i}. {word} "
            f"{probability * 100:.2f}%"
        )

In [ ]:
# ============================================================
# 17. GENERATE 5 WORDS
# ============================================================

def generate_words(
    text,
    num_words=5
):

    result = text
    generated = []

    for _ in range(num_words):

        predictions = predict_next_words(
            result,
            top_k=1
        )

        if not predictions:
            break

        next_word = predictions[0][0]

        if next_word in {
            "<PAD>",
            "<UNK>",
            "<START>"
        }:
            break

        generated.append(
            next_word
        )

        result = (
            result.rstrip()
            + " "
            + next_word
        )

    return result, generated


text = "วันนี้ฉันกำลังจะไป"

print_suggestions(
    text,
    top_k=5
)

result, generated = generate_words(
    text,
    num_words=5
)

print("\nGenerated 5 words:")
print(generated)

print("\nResult:")
print(result)

In [ ]:
# ============================================================
# 18. INTERACTIVE TEST
# ============================================================

print("Thai Next-Word Prediction")
print("พิมพ์ exit เพื่อออก\n")

while True:

    text = input(
        "Enter Thai sentence: "
    )

    if text.lower().strip() == "exit":
        break

    if not text.strip():
        continue

    print_suggestions(
        text,
        top_k=5
    )

    result, generated = generate_words(
        text,
        num_words=5
    )

    print("\nGenerate 5 words:")
    print(result)
    print()

In [ ]:
# ============================================================
# 19. SAVE KERAS + VOCABULARY
# ============================================================

MODEL_PATH = os.path.join(
    OUTPUT_DIR,
    "thai_next_word_transformer.keras"
)

VOCAB_PATH = os.path.join(
    OUTPUT_DIR,
    "thai_vocab.json"
)

model.save(MODEL_PATH)

with open(VOCAB_PATH, "w", encoding="utf-8") as f:
    json.dump(
        word_to_id,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved:")
print(MODEL_PATH)
print(VOCAB_PATH)

In [ ]:
# ============================================================
# 20. EXPORT TFLITE
# ============================================================

TFLITE_PATH = os.path.join(
    OUTPUT_DIR,
    "thai_next_word_transformer.tflite"
)

converter = tf.lite.TFLiteConverter.from_keras_model(model)

tflite_model = converter.convert()

with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_model)

size_mb = os.path.getsize(TFLITE_PATH) / (1024 ** 2)

print("TFLite:")
print(TFLITE_PATH)
print(f"Size: {size_mb:.2f} MB")

In [ ]:
# ============================================================
# 21. TEST TFLITE
# ============================================================

interpreter = tf.lite.Interpreter(
    model_path=TFLITE_PATH
)

interpreter.allocate_tensors()

input_details = (
    interpreter
    .get_input_details()
)

output_details = (
    interpreter
    .get_output_details()
)

print("Input:")
print(input_details)

print("\nOutput:")
print(output_details)

for x_batch, y_batch in train_ds.take(1):

    sample = x_batch[:1].numpy()

    interpreter.set_tensor(
        input_details[0]["index"],
        sample
    )

    interpreter.invoke()

    tflite_output = (
        interpreter
        .get_tensor(
            output_details[0]["index"]
        )
    )

    print(
        "\nTFLite output shape:",
        tflite_output.shape
    )

    print(
        "Predicted ID:",
        int(
            np.argmax(
                tflite_output[0]
            )
        )
    )

    break

## Important

For the first run, the notebook limits generated training examples to:

```python
MAX_TRAIN_SAMPLES = 1_000_000
MAX_VAL_SAMPLES = 100_000
```

This is intentional. Once the complete pipeline works, increase these values.

The notebook uses causal attention for next-word prediction rather than directly using bidirectional BERT. The provided course material distinguishes encoder-only BERT from decoder/autoregressive models; this implementation keeps the Transformer Encoder structure while applying a causal mask for the next-token task.